# DualPrompt Finetuning trên Kaggle
Notebook này thiết lập môi trường để finetune mô hình DualPrompt với dataset IP102.

In [ ]:
# Tự động phát hiện môi trường Kaggle và thiết lập dataset
import os

KAGGLE_INPUT_DIR = '/kaggle/input'
DATASET_DIR = os.path.join(KAGGLE_INPUT_DIR, 'datasets/nta212/ip102-for-object-detection')
os.environ['IP102_DATA_PATH'] = DATASET_DIR

# Copy model pretrained vào cache để timm không tải lại
os.system('mkdir -p ~/.cache/torch/hub/checkpoints/')
# Copy pretrain model path do người dùng chỉ định vào cache directory
os.system('cp /kaggle/input/models/fp3924/vit_base_patch16_224/pytorch/default/1/vit_base_patch16_224.pth ~/.cache/torch/hub/checkpoints/jx_vit_base_p16_224-80ecf9dd.pth')


In [ ]:
!pip install timm==0.4.12 scikit-learn pandas

In [ ]:
# Tự động clone code từ GitHub và chuyển thư mục làm việc
import os
REPO_URL = os.environ.get('IP102_CODE_REPO', 'https://github.com/nta2112/DualPromt-for-IP102.git')
REPO_DIR = '/kaggle/working/DualPromt-for-IP102'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Current working directory:', os.getcwd())

In [ ]:
# Chạy script training với 2 GPU T4 (Kaggle hỗ trợ 2 GPU)
# Sử dụng torchrun thay cho torch.distributed.launch đã bị deprecated
!torchrun --nproc_per_node=2 main.py \
    cifar100_dualprompt \
    --model vit_base_patch16_224 \
    --batch-size 24 \
    --dataset IP102 \
    --epochs 1 \
    --num_tasks 4 \
    --output_dir ./output

In [ ]:
# Hiển thị file results.csv
import pandas as pd
import glob

csv_files = glob.glob('./output/results.csv')
if csv_files:
    df = pd.read_csv(csv_files[0])
    display(df)
else:
    print('Không tìm thấy results.csv')